# Local 3-Stage Training Pipeline

**Stage 1 — Pretrain** on OpenCows2020 (46K cattle images, 186 individual cow IDs)  
**Stage 2 — Keypoint transfer** with AP-10K cattle keypoints as auxiliary loss  
**Stage 3 — Weight fine-tune** on local weight-labeled data (Kaggle 3102 + CID 2052)

**Why this beats ImageNet-only init:**  
- OpenCows gives cattle-specific texture, silhouette, and body proportion features  
- AP-10K keypoints force the backbone to localize anatomical landmarks (withers, hips, knees) — the same landmarks that correlate with body volume and weight  
- Result: faster convergence + lower MAPE on weight prediction

**Prerequisites:**  
Run `python download_datasets.py --all` from the project root first.

In [ ]:
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

import importlib
def ensure(pkg, import_as=None):
    try:
        importlib.import_module(import_as or pkg)
    except Exception:
        _pip(pkg)

ensure('albumentations')
ensure('pycocotools', import_as='pycocotools.coco')
ensure('timm')
ensure('scikit-image', import_as='skimage')

try:
    import tensorflow as _tf
    _v = tuple(int(x) for x in _tf.__version__.split('.')[:2])
    if _v < (2, 16):
        _pip('tensorflow>=2.16', '--upgrade-strategy', 'only-if-needed')
except ImportError:
    _pip('tensorflow[and-cuda]>=2.16')  # installs GPU support on Linux; on Windows use tensorflow-directml

import os, json, re, warnings, gc, time, math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S, MobileNetV3Large
import albumentations as A
warnings.filterwarnings('ignore')

print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

# Use mixed precision only if GPU is available
if tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision: enabled (GPU)')
else:
    print('No GPU — running on CPU (training will be slow; reduce epochs)')

In [ ]:
# ── Configuration — edit these paths before running ──────────────────────────
NOTEBOOK_DIR  = Path('.')
PROJECT_ROOT  = NOTEBOOK_DIR.parent.resolve()
DATA_DIR      = PROJECT_ROOT / 'local_datasets'
OUTPUT_DIR    = PROJECT_ROOT / 'local_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENCOWS_DIR      = DATA_DIR / 'opencows2020'
AP10K_DIR         = DATA_DIR / 'ap10k'
KAGGLE_DIR        = DATA_DIR / 'kaggle_cattle'

# Indian cattle dirs (breed images, NO weight labels — used for domain adaptation)
INDIAN_KAGGLE_DIR   = DATA_DIR / 'indian_cattle_kaggle'   # 2,427 images, 10 breeds (Gir, Sahiwal, Ongole …)
INDIAN_BOVINE_DIR   = DATA_DIR / 'indian_bovine_kaggle'   # ~500 images, multiple breeds
INDIAN_ROBOFLOW_DIR = DATA_DIR / 'indian_cows_roboflow'   # 1,847 images, 14 breeds
INDIAN_LABELED_DIR  = DATA_DIR / 'indian_cattle_labeled'  # YOUR collected weight-labeled Indian data

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE     = 384
MOBILE_SIZE  = 224
BATCH_SIZE   = 16      # reduce to 8 if OOM on local GPU
SEED         = 42
WEIGHT_MIN   = 50
WEIGHT_MAX   = 700
MEAS_DIM     = 14
META_DIM     = 2
AUX_DIM      = MEAS_DIM + META_DIM

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')

checks = [
    ('OpenCows2020',         OPENCOWS_DIR),
    ('AP-10K',               AP10K_DIR),
    ('Kaggle weight',        KAGGLE_DIR),
    ('Indian Cattle (Kaggle)',   INDIAN_KAGGLE_DIR),
    ('Indian Bovine (Kaggle)',   INDIAN_BOVINE_DIR),
    ('Indian Cows (Roboflow)',   INDIAN_ROBOFLOW_DIR),
    ('Indian labeled (yours)',   INDIAN_LABELED_DIR),
]
for name, d in checks:
    n = sum(1 for _ in d.rglob('*.jpg')) if d.exists() else 0
    tag = f'{n} images' if n else 'NOT FOUND'
    print(f'  {name:28s}: {tag}')

## Shared Utilities
Augmentation pipelines and mask feature extractor — shared across all 3 stages.

In [ ]:
# ── Augmentation ──────────────────────────────────────────────────────────────
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

def make_aug(img_size, train=True):
    if train:
        return A.Compose([
            A.LongestMaxSize(max_size=img_size),
            A.PadIfNeeded(img_size, img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.HueSaturationValue(10, 20, 10, p=0.4),
            A.GaussNoise(var_limit=(10, 50), p=0.3),
            A.Blur(blur_limit=3, p=0.2),
            A.ShiftScaleRotate(0.05, 0.1, 5, border_mode=cv2.BORDER_CONSTANT, p=0.4),
            A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
            A.CLAHE(p=0.2),
            A.Normalize(mean=IMG_MEAN, std=IMG_STD),
        ])
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ])

# ── Mask feature extractor (14-dim) — same as notebook 02 ────────────────────
from skimage.morphology import skeletonize

def extract_mask_features(mask_path_or_img):
    if isinstance(mask_path_or_img, (str, Path)):
        mask = cv2.imread(str(mask_path_or_img), cv2.IMREAD_GRAYSCALE)
    else:
        mask = mask_path_or_img
    if mask is None:
        return np.zeros(14, dtype=np.float32)
    H, W = mask.shape
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros(14, dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perim = cv2.arcLength(cnt, True)
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    x, y, w, h = cv2.boundingRect(cnt)
    solidity    = area / (hull_area + 1e-6)
    compactness = (4 * np.pi * area) / (perim**2 + 1e-6)
    aspect      = w / (h + 1e-6)
    extent      = area / (w * h + 1e-6)
    fill        = area / (H * W)
    el_ratio = 0.0
    if len(cnt) >= 5:
        _, (ma, mi), _ = cv2.fitEllipse(cnt)
        el_ratio = ma / (mi + 1e-6)
    roi = binary[y:y+h, x:x+w]
    roi_s = cv2.resize(roi, (64, 64)) > 127
    skel_len = skeletonize(roi_s).sum() / 64
    mid_y = H // 2
    top_fill = binary[:mid_y, :].sum() / 255 / (H/2 * W + 1e-6)
    bot_fill = binary[mid_y:, :].sum() / 255 / (H/2 * W + 1e-6)
    return np.array([
        fill, w/W, h/H, aspect,
        solidity, compactness, extent, el_ratio,
        skel_len, top_fill, bot_fill,
        perim / (W + H + 1e-6),
        hull_area / (W * H + 1e-6),
        (area - hull_area) / (hull_area + 1e-6),
    ], dtype=np.float32)

# ── Shared metrics ────────────────────────────────────────────────────────────
WEIGHT_MEAN = WEIGHT_STD = None  # set in Stage 3

def mape_metric(y_true, y_pred):
    y_t = y_true * WEIGHT_STD + WEIGHT_MEAN
    y_p = y_pred * WEIGHT_STD + WEIGHT_MEAN
    return tf.reduce_mean(tf.abs(y_t - y_p) / (y_t + 1e-6)) * 100

def mae_kg_metric(y_true, y_pred):
    y_t = y_true * WEIGHT_STD + WEIGHT_MEAN
    y_p = y_pred * WEIGHT_STD + WEIGHT_MEAN
    return tf.reduce_mean(tf.abs(y_t - y_p))

print('Shared utilities ready.')

## Stage 1 — OpenCows2020 Pretraining

**Task:** 186-class cow identity classification.  
**Why:** Forces the backbone to learn cattle-specific body proportions, coat patterns, and silhouettes — features that directly predict body volume and weight.  
**Output:** `local_output/backbone_opencows.weights.h5`

In [ ]:
# ── OpenCows2020 dataset loader ────────────────────────────────────────────────
# Expected structure after extraction:
#   opencows2020/recognition/data/<cow_id>/<image>.jpg
# If Bristol releases a different structure, adjust RECOG_ROOT below.

RECOG_ROOT = OPENCOWS_DIR / 'recognition' / 'data'
if not RECOG_ROOT.exists():
    # Fallback: any sub-directory that contains numbered/named subdirs of images
    candidates = [d for d in OPENCOWS_DIR.rglob('*') if d.is_dir() and
                  sum(1 for _ in d.glob('*.jpg')) > 5]
    if candidates:
        RECOG_ROOT = max(candidates, key=lambda d: len(list(d.glob('*.jpg')))).parent
        print(f'[opencows] Auto-detected root: {RECOG_ROOT}')
    else:
        print('[opencows] ERROR: No images found. Run: python download_datasets.py --opencows')

# Build (path, label_idx) pairs
cow_dirs = sorted([d for d in RECOG_ROOT.iterdir() if d.is_dir()]) if RECOG_ROOT.exists() else []
N_CLASSES = len(cow_dirs)
print(f'OpenCows2020: {N_CLASSES} cow identities')

oc_records = []
for idx, cow_dir in enumerate(cow_dirs):
    for img_path in cow_dir.glob('*.jpg'):
        oc_records.append({'path': str(img_path), 'label': idx})

oc_df = pd.DataFrame(oc_records)
oc_train, oc_val = train_test_split(oc_df, test_size=0.1, stratify=oc_df['label'], random_state=SEED)
print(f'  Train: {len(oc_train)}  Val: {len(oc_val)}')

# TF Dataset
train_aug_oc = make_aug(IMG_SIZE, train=True)
val_aug_oc   = make_aug(IMG_SIZE, train=False)

def oc_gen(df, augment):
    aug = train_aug_oc if augment else val_aug_oc
    rows = df.to_dict('records')
    if augment:
        import random; random.seed(SEED); random.shuffle(rows)
    def _gen():
        for r in rows:
            img = cv2.imread(r['path'])
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = aug(image=img)['image'].astype(np.float32)
            label = np.int32(r['label'])
            yield img, label
    return _gen

def make_oc_ds(df, augment, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_generator(
        oc_gen(df, augment),
        output_signature=(
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        )
    )
    if not augment: ds = ds.cache()
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

oc_train_ds = make_oc_ds(oc_train, augment=True)
oc_val_ds   = make_oc_ds(oc_val,   augment=False)
print('OpenCows2020 TF datasets ready.')

In [ ]:
# ── Stage 1 model: EfficientNetV2-S + identity classification head ────────────
BACKBONE_SAVE  = OUTPUT_DIR / 'backbone_opencows.weights.h5'
PRETRAIN_SAVE  = OUTPUT_DIR / 'pretrain_opencows_model.keras'

def build_pretrain_model(num_classes=N_CLASSES):
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
    backbone = EfficientNetV2S(include_top=False, weights='imagenet', name='backbone')
    x = backbone(inp)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax', dtype='float32', name='identity')(x)
    return Model(inputs=inp, outputs=out, name='opencows_pretrain')

if BACKBONE_SAVE.exists():
    print(f'[Stage 1] Already complete — backbone weights at {BACKBONE_SAVE}')
    print('Skipping Stage 1 training. Delete the file to retrain.')
    STAGE1_DONE = True
else:
    pretrain_model = build_pretrain_model()
    pretrain_model.summary(line_length=80)
    STAGE1_DONE = False

In [ ]:
# ── Stage 1 training ──────────────────────────────────────────────────────────
if not STAGE1_DONE:
    # Phase A: frozen backbone, train head only (fast)
    pretrain_model.get_layer('backbone').trainable = False
    pretrain_model.compile(
        optimizer=keras.optimizers.AdamW(1e-3, weight_decay=1e-4),
        loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    print('[Stage 1a] Frozen backbone — training head for 10 epochs ...')
    hist_1a = pretrain_model.fit(
        oc_train_ds, validation_data=oc_val_ds, epochs=10, verbose=1,
        callbacks=[keras.callbacks.EarlyStopping('val_accuracy', patience=4, restore_best_weights=True)]
    )

    # Phase B: unfreeze top 40% of backbone, fine-tune
    backbone_layer = pretrain_model.get_layer('backbone')
    n_layers = len(backbone_layer.layers)
    for lyr in backbone_layer.layers[int(n_layers * 0.6):]:
        lyr.trainable = True

    total_steps = 25 * math.ceil(len(oc_train) / BATCH_SIZE)
    warmup_steps = max(1, int(0.05 * total_steps))

    class WarmupCosine(keras.optimizers.schedules.LearningRateSchedule):
        def __init__(self, base_lr, total, warmup):
            self.base_lr, self.total, self.warmup = base_lr, total, warmup
        def __call__(self, step):
            step = tf.cast(step, tf.float32)
            warmup_lr = self.base_lr * (step / self.warmup)
            cos_lr = self.base_lr * 0.5 * (1 + tf.cos(math.pi * (step - self.warmup) / (self.total - self.warmup)))
            return tf.where(step < self.warmup, warmup_lr, cos_lr)
        def get_config(self):
            return {'base_lr': self.base_lr, 'total': self.total, 'warmup': self.warmup}

    pretrain_model.compile(
        optimizer=keras.optimizers.AdamW(
            WarmupCosine(3e-4, total_steps, warmup_steps), weight_decay=1e-4, clipnorm=1.0
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    print('[Stage 1b] Top-40% backbone fine-tune — 25 epochs ...')
    hist_1b = pretrain_model.fit(
        oc_train_ds, validation_data=oc_val_ds, epochs=25, verbose=1,
        callbacks=[
            keras.callbacks.ModelCheckpoint(str(PRETRAIN_SAVE), save_best_only=True, monitor='val_accuracy'),
            keras.callbacks.EarlyStopping('val_accuracy', patience=7, restore_best_weights=True),
        ]
    )

    # Save backbone weights only (discards classification head)
    pretrain_model.get_layer('backbone').save_weights(str(BACKBONE_SAVE))
    print(f'[Stage 1] Backbone weights saved → {BACKBONE_SAVE}')

    val_acc = max(hist_1b.history['val_accuracy'])
    print(f'[Stage 1] Best val accuracy: {val_acc:.3f} (186-class cow ID)')
    STAGE1_DONE = True
    del pretrain_model; gc.collect()

## Stage 1.5 — Indian Breed Domain Adaptation

**The problem:** OpenCows2020 contains only Holstein Friesian cattle (European Bos taurus).  
Indian breeds — Gir, Sahiwal, Ongole, Kankrej, Hariana — are Bos indicus. They have:
- A distinctive hump (Brahman hump)  
- Longer ears and dewlap  
- Different body proportions and colour patterns  

Without this step the backbone has never seen an Indian cow. This stage fine-tunes it on  
~4,300 Indian cattle images (Kaggle + Roboflow combined) using breed classification,  
so it learns Indian body structure before the weight fine-tuning step.

**Output:** `local_output/backbone_indian.weights.h5`  
If none of the Indian datasets are present this stage is skipped automatically.

In [ ]:
INDIAN_BACKBONE_SAVE = OUTPUT_DIR / 'backbone_indian.weights.h5'

# ── Collect images from all available Indian cattle datasets ──────────────────
# Each dataset may be organised differently; we unify them into (path, breed_label) pairs.

def _scan_dir_by_subfolder(root, existing_labels=None):
    """Images in subfolders → each subfolder = one breed class."""
    records = []
    if not root.exists():
        return records, existing_labels or {}
    labels = existing_labels or {}
    for sub in sorted(root.iterdir()):
        if not sub.is_dir():
            continue
        breed = sub.name.lower().strip()
        if breed not in labels:
            labels[breed] = len(labels)
        idx = labels[breed]
        for img in sub.rglob('*.jpg'):
            records.append({'path': str(img), 'breed': breed, 'label': idx})
    return records, labels

def _scan_roboflow(root, existing_labels=None):
    """Roboflow COCO layout: images/ + annotations/*.json with category names."""
    records = []
    labels = existing_labels or {}
    if not root.exists():
        return records, labels
    ann_files = list(root.rglob('*.json'))
    img_dirs  = list(root.rglob('images'))
    if not ann_files:
        # Fallback: treat sub-folders as breeds
        return _scan_dir_by_subfolder(root, labels)
    for jp in ann_files:
        try:
            data = json.load(open(jp))
        except Exception:
            continue
        if 'categories' not in data:
            continue
        cat_map = {c['id']: c['name'].lower() for c in data['categories']}
        img_map = {img['id']: img['file_name'] for img in data.get('images', [])}
        ann_map = defaultdict(list)
        for a in data.get('annotations', []):
            ann_map[a['image_id']].append(a['category_id'])
        for img_id, fname in img_map.items():
            cats = ann_map.get(img_id, [])
            if not cats:
                continue
            breed = cat_map.get(cats[0], 'unknown')
            if breed not in labels:
                labels[breed] = len(labels)
            # Find image file
            found = []
            for img_dir in img_dirs:
                found = list(img_dir.rglob(Path(fname).name))
                if found:
                    break
            if not found:
                found = list(root.rglob(Path(fname).name))
            if found:
                records.append({'path': str(found[0]), 'breed': breed, 'label': labels[breed]})
    return records, labels

# Scan each Indian dataset
all_indian, breed_labels = [], {}
r1, breed_labels = _scan_dir_by_subfolder(INDIAN_KAGGLE_DIR,   breed_labels)
r2, breed_labels = _scan_dir_by_subfolder(INDIAN_BOVINE_DIR,   breed_labels)
r3, breed_labels = _scan_roboflow(INDIAN_ROBOFLOW_DIR,          breed_labels)
all_indian = r1 + r2 + r3

N_INDIAN_CLASSES = len(breed_labels)
print(f'Indian cattle images   : {len(all_indian)}')
print(f'Breed classes found    : {N_INDIAN_CLASSES}')
if breed_labels:
    print('  Breeds:', ', '.join(sorted(breed_labels.keys())))

STAGE15_SKIP = len(all_indian) < 50
if STAGE15_SKIP:
    print('\n[Stage 1.5] No Indian cattle images found — skipping.')
    print('  Run: python download_datasets.py --indian')
    import shutil
    src = BACKBONE_SAVE if BACKBONE_SAVE.exists() else None
    if src:
        shutil.copy(src, INDIAN_BACKBONE_SAVE)
        print(f'  Stage 1 backbone copied → {INDIAN_BACKBONE_SAVE}')
elif INDIAN_BACKBONE_SAVE.exists():
    print(f'\n[Stage 1.5] Already complete — {INDIAN_BACKBONE_SAVE}')
else:
    ind_df = pd.DataFrame(all_indian)
    ind_train, ind_val = train_test_split(
        ind_df, test_size=0.1,
        stratify=ind_df['label'] if ind_df['label'].value_counts().min() >= 2 else None,
        random_state=SEED
    )
    print(f'  Train: {len(ind_train)}  Val: {len(ind_val)}')

    aug_ind_tr = make_aug(IMG_SIZE, train=True)
    aug_ind_vl = make_aug(IMG_SIZE, train=False)

    def ind_gen(df, augment):
        aug = aug_ind_tr if augment else aug_ind_vl
        rows = df.to_dict('records')
        if augment:
            import random; random.seed(SEED); random.shuffle(rows)
        def _gen():
            for r in rows:
                img = cv2.imread(r['path'])
                if img is None: continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = aug(image=img)['image'].astype(np.float32)
                yield img, np.int32(r['label'])
        return _gen

    def make_ind_ds(df, augment):
        ds = tf.data.Dataset.from_generator(
            ind_gen(df, augment),
            output_signature=(
                tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32),
                tf.TensorSpec((), tf.int32),
            )
        )
        if not augment: ds = ds.cache()
        return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    ind_train_ds = make_ind_ds(ind_train, augment=True)
    ind_val_ds   = make_ind_ds(ind_val,   augment=False)

    # Load backbone pretrained from Stage 1 (or ImageNet if Stage 1 skipped)
    ind_inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
    ind_bb  = EfficientNetV2S(include_top=False, weights=None, name='backbone')
    if BACKBONE_SAVE.exists():
        ind_bb.load_weights(str(BACKBONE_SAVE))
        print('  Loaded Stage 1 backbone weights (OpenCows2020 pretrained)')
    else:
        ind_bb = EfficientNetV2S(include_top=False, weights='imagenet', name='backbone')
        print('  Stage 1 backbone not found — using ImageNet init')

    # Freeze bottom 60%, train top 40%
    n = len(ind_bb.layers)
    for lyr in ind_bb.layers[:int(n * 0.6)]:
        lyr.trainable = False

    x = ind_bb(ind_inp)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.2)(x)
    ind_out = layers.Dense(N_INDIAN_CLASSES, activation='softmax', dtype='float32', name='breed')(x)
    ind_model = Model(inputs=ind_inp, outputs=ind_out, name='indian_domain_adapt')

    ind_model.compile(
        optimizer=keras.optimizers.AdamW(5e-4, weight_decay=1e-4, clipnorm=1.0),
        loss=keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

    print('\n[Stage 1.5] Training Indian breed classifier (20 epochs) ...')
    hist_15 = ind_model.fit(
        ind_train_ds, validation_data=ind_val_ds, epochs=20, verbose=1,
        callbacks=[
            keras.callbacks.EarlyStopping('val_accuracy', patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_accuracy', factor=0.5, patience=3, mode='max'),
        ]
    )

    ind_model.get_layer('backbone').save_weights(str(INDIAN_BACKBONE_SAVE))
    best_acc = max(hist_15.history['val_accuracy'])
    print(f'[Stage 1.5] Best val accuracy: {best_acc:.3f}  Backbone → {INDIAN_BACKBONE_SAVE}')
    del ind_model; gc.collect()

## Stage 2 — AP-10K Cattle Keypoint Transfer

**Task:** Predict 17 cattle body keypoints (COCO format) alongside identity.  
**Why:** Keypoints pinpoint withers, hips, knees — anatomical landmarks whose distances directly correlate with body length, height, and volume — the geometric basis of weight estimation.  
**Output:** `local_output/backbone_ap10k.weights.h5`

In [ ]:
# ── AP-10K cattle subset loader ───────────────────────────────────────────────
# AP-10K uses COCO keypoint format:
#   data/images/train/ val/ test/
#   data/annotations/ap10k-train-split1.json etc.

AP10K_ANN_DIR  = AP10K_DIR / 'data' / 'annotations'
AP10K_IMG_DIR  = AP10K_DIR / 'data' / 'images'

# Fallback: search for annotation JSONs anywhere under AP10K_DIR
if not AP10K_ANN_DIR.exists():
    jsons = list(AP10K_DIR.rglob('*.json'))
    if jsons:
        AP10K_ANN_DIR = jsons[0].parent
        AP10K_IMG_DIR = AP10K_ANN_DIR.parent / 'images'
        print(f'[ap10k] Auto-detected annotation dir: {AP10K_ANN_DIR}')
    else:
        print('[ap10k] No annotations found. Run: python download_datasets.py --ap10k')

N_KEYPOINTS = 17
CATTLE_KP_SAVE = OUTPUT_DIR / 'backbone_ap10k.weights.h5'

def load_ap10k_cattle(ann_dir, img_root):
    """Load cattle-subset images + keypoints from AP-10K COCO annotations."""
    records = []
    for json_path in sorted(ann_dir.glob('*.json')):
        try:
            with open(json_path) as f:
                data = json.load(f)
        except Exception:
            continue
        if 'annotations' not in data:
            continue

        # Find cattle category ID (look for 'cow' or 'cattle')
        cattle_ids = set()
        for cat in data.get('categories', []):
            if any(k in cat.get('name', '').lower() for k in ['cow', 'cattle', 'bovine']):
                cattle_ids.add(cat['id'])
        if not cattle_ids:
            continue

        img_map = {img['id']: img for img in data.get('images', [])}
        for ann in data.get('annotations', []):
            if ann.get('category_id') not in cattle_ids:
                continue
            kps = ann.get('keypoints', [])
            if len(kps) < N_KEYPOINTS * 3:
                continue
            img_info = img_map.get(ann['image_id'])
            if img_info is None:
                continue
            fname = img_info['file_name']
            # Try to find image under img_root
            found = list(img_root.rglob(Path(fname).name)) if img_root.exists() else []
            if not found:
                continue
            kp_array = np.array(kps, dtype=np.float32).reshape(N_KEYPOINTS, 3)  # x, y, v
            records.append({
                'path': str(found[0]),
                'keypoints': kp_array,
                'width': img_info.get('width', 640),
                'height': img_info.get('height', 480),
            })
    return records

ap10k_records = load_ap10k_cattle(AP10K_ANN_DIR, AP10K_IMG_DIR) if AP10K_ANN_DIR.exists() else []
print(f'AP-10K cattle annotations: {len(ap10k_records)}')

STAGE2_SKIP = len(ap10k_records) < 50
if STAGE2_SKIP:
    print('[Stage 2] Not enough AP-10K data — will skip Stage 2 and use Stage 1 backbone directly.')

In [ ]:
# ── Stage 2: keypoint transfer ────────────────────────────────────────────────
if CATTLE_KP_SAVE.exists():
    print(f'[Stage 2] Already complete — backbone at {CATTLE_KP_SAVE}')
    STAGE2_DONE = True
elif STAGE2_SKIP:
    # Copy Stage 1 backbone as Stage 2 backbone so Stage 3 can proceed unchanged
    import shutil
    if BACKBONE_SAVE.exists():
        shutil.copy(BACKBONE_SAVE, CATTLE_KP_SAVE)
        print('[Stage 2] Skipped — Stage 1 backbone copied as Stage 2 backbone.')
    STAGE2_DONE = True
else:
    # ── Build keypoint model on top of pretrained backbone ────────────────────
    def build_keypoint_model(weights_path, num_kp=N_KEYPOINTS):
        inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
        backbone = EfficientNetV2S(include_top=False, weights=None, name='backbone')
        if weights_path and Path(weights_path).exists():
            backbone.load_weights(str(weights_path))
            print(f'  Loaded backbone from {weights_path}')
        # Freeze all but top 30%
        n = len(backbone.layers)
        for lyr in backbone.layers[:int(n * 0.7)]:
            lyr.trainable = False
        x = backbone(inp)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(512, activation='swish')(x)
        x = layers.Dropout(0.3)(x)
        # Keypoint coordinates: num_kp * 2 (normalized x, y — only visible keypoints matter)
        out = layers.Dense(num_kp * 2, dtype='float32', name='keypoints')(x)
        return Model(inputs=inp, outputs=out, name='kp_model')

    kp_model = build_keypoint_model(BACKBONE_SAVE)

    # Dataset
    kp_train, kp_val = train_test_split(ap10k_records, test_size=0.15, random_state=SEED)
    aug_kp_train = make_aug(IMG_SIZE, train=True)
    aug_kp_val   = make_aug(IMG_SIZE, train=False)

    def kp_gen(records, augment):
        aug = aug_kp_train if augment else aug_kp_val
        def _gen():
            for r in records:
                img = cv2.imread(r['path'])
                if img is None: continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                H, W = img.shape[:2]
                img_out = aug(image=img)['image'].astype(np.float32)
                kps = r['keypoints']  # shape (17, 3): x, y, visibility
                # Normalize coordinates to [0, 1], zero out invisible keypoints
                xy = np.zeros(N_KEYPOINTS * 2, dtype=np.float32)
                for i, (kx, ky, vis) in enumerate(kps):
                    if vis > 0:
                        xy[i*2]   = kx / (W + 1e-6)
                        xy[i*2+1] = ky / (H + 1e-6)
                yield img_out, xy
        return _gen

    def make_kp_ds(records, augment):
        ds = tf.data.Dataset.from_generator(
            kp_gen(records, augment),
            output_signature=(
                tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32),
                tf.TensorSpec((N_KEYPOINTS * 2,), tf.float32),
            )
        )
        if not augment: ds = ds.cache()
        return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    kp_train_ds = make_kp_ds(kp_train, augment=True)
    kp_val_ds   = make_kp_ds(kp_val,   augment=False)

    kp_model.compile(
        optimizer=keras.optimizers.AdamW(1e-4, weight_decay=1e-4),
        loss='mse',
        metrics=['mae']
    )
    print('[Stage 2] Training keypoint model (20 epochs) ...')
    hist_2 = kp_model.fit(
        kp_train_ds, validation_data=kp_val_ds, epochs=20, verbose=1,
        callbacks=[
            keras.callbacks.EarlyStopping('val_loss', patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=3),
        ]
    )

    kp_model.get_layer('backbone').save_weights(str(CATTLE_KP_SAVE))
    print(f'[Stage 2] Keypoint-tuned backbone saved → {CATTLE_KP_SAVE}')
    STAGE2_DONE = True
    del kp_model; gc.collect()

## Stage 3 — Weight Fine-tuning

Train **three variants** in one pass so we can directly measure the pretraining benefit:

| Variant | Backbone init | Expected MAPE |
|---------|--------------|---------------|
| A — Baseline | ImageNet weights | highest |
| B — OpenCows | Stage 1 pretrained | lower |
| C — +AP-10K  | Stage 2 pretrained | lowest |

All three share the same multi-input architecture: EfficientNetV2-S backbone + mask spatial branch + 16-dim AUX (14 body measurements + age + sex).

In [ ]:
# ── Load local weight data ────────────────────────────────────────────────────
import re

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def parse_filename(stem):
    m = re.match(r'[\d\.]+_s_(\d+)_([\d\.]+)_([MFmf])', stem)
    if m:
        w = float(m.group(1))
        if WEIGHT_MIN <= w <= WEIGHT_MAX:
            return w, float(m.group(2)), m.group(3).upper()
    m2 = re.search(r'_(\d{2,4})_', stem)
    if m2:
        v = int(m2.group(1))
        if WEIGHT_MIN <= v <= WEIGHT_MAX:
            return float(v), np.nan, ''
    return np.nan, np.nan, ''

def discover_local(root):
    records = []
    for jp in root.rglob('*.json'):
        try:
            data = json.load(open(jp))
        except Exception: continue
        if 'images' not in data: continue
        ann_map = defaultdict(list)
        for a in data.get('annotations', []): ann_map[a['image_id']].append(a)
        for img in data.get('images', []):
            fname = img.get('file_name', '')
            candidates = [c for c in root.rglob(Path(fname).name) if '___fuse' not in c.name]
            if not candidates: continue
            fn_w, fn_age, fn_sex = parse_filename(Path(fname).stem)
            w_raw = img.get('weight', img.get('weight_kg', np.nan))
            try: w = float(w_raw)
            except (TypeError, ValueError): w = np.nan
            if np.isnan(w): w = fn_w
            anns = ann_map.get(img['id'], [])
            records.append({
                'full_path': str(candidates[0]), 'weight_kg': w,
                'age': fn_age, 'sex': fn_sex,
                'has_seg': any('segmentation' in a and a['segmentation'] for a in anns),
            })
    covered = {r['full_path'] for r in records}
    for f in root.rglob('*'):
        if f.is_file() and f.suffix.lower() in {'.jpg','.jpeg'} and str(f) not in covered and '___fuse' not in f.name:
            w, age, sex = parse_filename(f.stem)
            if not np.isnan(w):
                records.append({'full_path': str(f), 'weight_kg': w, 'age': age, 'sex': sex, 'has_seg': False})
    # Also check for a weights.csv in the root
    for csv_path in root.glob('*.csv'):
        try:
            csv_df = pd.read_csv(csv_path)
            col_map = {}
            for c in csv_df.columns:
                cl = c.lower()
                if 'weight' in cl: col_map[c] = 'weight_kg'
                elif 'age' in cl: col_map[c] = 'age'
                elif 'sex' in cl or 'gender' in cl: col_map[c] = 'sex'
                elif any(k in cl for k in ['file','image','name']): col_map[c] = 'file_name'
            csv_df = csv_df.rename(columns=col_map)
            if 'file_name' in csv_df.columns and 'weight_kg' in csv_df.columns:
                for _, row in csv_df.iterrows():
                    found = list(root.rglob(Path(str(row['file_name'])).name))
                    if found:
                        records.append({
                            'full_path': str(found[0]),
                            'weight_kg': float(row['weight_kg']) if pd.notna(row.get('weight_kg')) else np.nan,
                            'age': row.get('age', np.nan), 'sex': row.get('sex', ''), 'has_seg': False,
                        })
        except Exception: pass
    return pd.DataFrame(records).drop_duplicates('full_path').reset_index(drop=True)

dfs = []
# Standard weight datasets
for name, d in [('kaggle', KAGGLE_DIR), ('cid', DATA_DIR / 'cid_cattle')]:
    if d.exists() and any(d.rglob('*.jpg')):
        sub = discover_local(d)
        sub['source'] = name
        dfs.append(sub)
        print(f'  {name:20s}: {len(sub)} records ({sub.weight_kg.notna().sum()} labeled)')

# Your Indian cattle labeled data — highest priority, shown separately
if INDIAN_LABELED_DIR.exists() and any(INDIAN_LABELED_DIR.rglob('*.jpg')):
    ind_labeled = discover_local(INDIAN_LABELED_DIR)
    ind_labeled['source'] = 'indian_labeled'
    n_labeled = ind_labeled.weight_kg.notna().sum()
    dfs.append(ind_labeled)
    print(f'  {"indian_labeled":20s}: {len(ind_labeled)} records ({n_labeled} labeled) ← YOUR DATA')
else:
    print(f'  {"indian_labeled":20s}: NOT FOUND')
    print('    → Collect images + weights following collect_indian_data/GUIDE.md')
    print('    → Save to local_datasets/indian_cattle_labeled/')

if not dfs:
    raise RuntimeError(
        'No weight-labeled images found!\n'
        'Run: python download_datasets.py --kaggle\n'
        'Or copy your Kaggle/CID data into local_datasets/kaggle_cattle/'
    )

df_all = pd.concat(dfs, ignore_index=True)
df_labeled = df_all.dropna(subset=['weight_kg']).copy()
df_labeled = df_labeled[
    (df_labeled.weight_kg >= WEIGHT_MIN) & (df_labeled.weight_kg <= WEIGHT_MAX)
].reset_index(drop=True)

print(f'\nTotal weight-labeled: {len(df_labeled)}')
print(df_labeled.groupby('source')['weight_kg'].describe()[['count','mean','min','max']].round(1))

In [ ]:
# ── Train/val/test split + weight normalization ────────────────────────────────
df_labeled['weight_bin'] = pd.qcut(df_labeled.weight_kg, q=10, labels=False, duplicates='drop')
train_df, temp_df = train_test_split(df_labeled, test_size=0.2, random_state=SEED, stratify=df_labeled.weight_bin)
val_df, test_df   = train_test_split(temp_df,    test_size=0.5, random_state=SEED, stratify=temp_df.weight_bin)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

WEIGHT_MEAN = float(train_df.weight_kg.mean())
WEIGHT_STD  = float(train_df.weight_kg.std())
print(f'Normalization: mean={WEIGHT_MEAN:.1f} kg  std={WEIGHT_STD:.1f} kg')

# Mask index for O(1) lookup
print('Building mask index ...')
_mask_index = {}
for root in [KAGGLE_DIR, DATA_DIR / 'cid_cattle']:
    if root.exists():
        for f in root.rglob('*.png'):
            if '___fuse' not in f.name:
                _mask_index[f.stem] = str(f)
print(f'Mask index: {len(_mask_index)} PNG files')

def find_mask(img_path):
    p = Path(img_path)
    ann_dir = p.parent.parent / 'annotations'
    if ann_dir.exists():
        c = ann_dir / (p.stem + '.png')
        if c.exists() and '___fuse' not in c.name: return str(c)
    if p.stem in _mask_index: return _mask_index[p.stem]
    return None

df_labeled['mask_path'] = df_labeled.full_path.apply(find_mask)
n_masks = df_labeled.mask_path.notna().sum()
USE_MASKS = n_masks > len(df_labeled) * 0.5
print(f'Masks available: {n_masks}/{len(df_labeled)} ({n_masks/len(df_labeled)*100:.1f}%)  USE_MASKS={USE_MASKS}')

In [ ]:
# ── Weight model definition ────────────────────────────────────────────────────
# Architecture: EfficientNetV2-S (RGB stream) + Mask CNN branch + AUX MLP
# Identical to notebook 02 — only the backbone init changes across variants.

def se_block(x, ratio=16):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D()(x)
    s = layers.Dense(c // ratio, activation='swish')(s)
    s = layers.Dense(c, activation='sigmoid')(s)
    s = layers.Reshape((1, 1, c))(s)
    return layers.Multiply()([x, s])

def build_weight_model(backbone_weights_path=None, freeze_fraction=0.7):
    """
    backbone_weights_path:
      None            → ImageNet init (Variant A)
      BACKBONE_SAVE   → OpenCows pretrained (Variant B)
      CATTLE_KP_SAVE  → +AP-10K keypoints (Variant C)
    """
    inp_img = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 4), name='image')
    inp_aux = layers.Input(shape=(AUX_DIM,), name='aux')

    # RGB stream
    rgb = layers.Lambda(lambda x: x[:, :, :, :3], name='rgb_split')(inp_img)
    backbone = EfficientNetV2S(include_top=False, weights='imagenet' if backbone_weights_path is None else None, name='backbone')
    if backbone_weights_path and Path(backbone_weights_path).exists():
        backbone.load_weights(str(backbone_weights_path))
    # Freeze bottom fraction of backbone
    n = len(backbone.layers)
    for lyr in backbone.layers[:int(n * freeze_fraction)]:
        lyr.trainable = False
    rgb_pool = layers.GlobalAveragePooling2D()(backbone(rgb))

    # Mask spatial stream
    mask_ch = layers.Lambda(lambda x: x[:, :, :, 3:4], name='mask_split')(inp_img)
    m = layers.Conv2D(32, 3, padding='same', activation='swish')(mask_ch)
    m = layers.MaxPooling2D(2)(m)
    m = layers.Conv2D(64, 3, padding='same', activation='swish')(m)
    m = se_block(m)
    m = layers.MaxPooling2D(2)(m)
    m = layers.Conv2D(128, 3, padding='same', activation='swish')(m)
    m = se_block(m)
    mask_pool = layers.GlobalAveragePooling2D()(m)

    # AUX MLP
    a = layers.Dense(64, activation='swish')(inp_aux)
    a = layers.BatchNormalization()(a)
    a = layers.Dense(64, activation='swish')(a)

    # Fusion
    fused = layers.Concatenate()([rgb_pool, mask_pool, a])
    fused = layers.BatchNormalization()(fused)
    fused = layers.Dense(512, activation='swish')(fused)
    fused = layers.Dropout(0.3)(fused)
    fused = layers.Dense(256, activation='swish')(fused)
    fused = layers.Dropout(0.2)(fused)
    fused = layers.Dense(64, activation='swish')(fused)
    out = layers.Dense(1, dtype='float32', name='weight')(fused)

    return Model(inputs=[inp_img, inp_aux], outputs=out)

print('Weight model builder defined.')

In [ ]:
# ── Weight dataset ────────────────────────────────────────────────────────────
train_aug_w = make_aug(IMG_SIZE, train=True)
val_aug_w   = make_aug(IMG_SIZE, train=False)

def load_weight_sample(row, augment=True):
    img = cv2.imread(row['full_path'])
    if img is None:
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mask_feats = np.zeros(MEAS_DIM, dtype=np.float32)
    mask_img = None
    if USE_MASKS and row.get('mask_path'):
        m = cv2.imread(str(row['mask_path']), cv2.IMREAD_GRAYSCALE)
        if m is not None:
            mask_feats = extract_mask_features(m)
            mask_img = cv2.resize(m, (img.shape[1], img.shape[0]))

    aug = train_aug_w if augment else val_aug_w
    if mask_img is not None:
        result = aug(image=img, mask=mask_img)
        img_out = result['image']
        mask_out = (result['mask'].astype(np.float32) / 255.0)
    else:
        img_out = aug(image=img)['image']
        mask_out = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)

    mask_ch = mask_out[:, :, np.newaxis] if mask_out.ndim == 2 else mask_out
    combined = np.concatenate([img_out, mask_ch], axis=-1).astype(np.float32)

    age_val = row.get('age', np.nan)
    meta = np.array([
        (float(age_val) if pd.notna(age_val) else 0.0) / 10.0,
        1.0 if str(row.get('sex', '')).lower() in ['m', 'male', 'bull'] else 0.0,
    ], dtype=np.float32)
    aux = np.concatenate([mask_feats, meta])
    weight_norm = (float(row['weight_kg']) - WEIGHT_MEAN) / WEIGHT_STD
    return combined, aux, weight_norm

def make_weight_ds(df_split, augment, batch_size=BATCH_SIZE):
    rows = df_split.to_dict('records')
    if augment:
        import random; random.seed(SEED); random.shuffle(rows)
    def _gen():
        for r in rows:
            try:
                img, aux, w = load_weight_sample(r, augment=augment)
                yield {'image': img, 'aux': aux}, np.float32(w)
            except Exception:
                pass
    ds = tf.data.Dataset.from_generator(
        _gen,
        output_signature=(
            {'image': tf.TensorSpec((IMG_SIZE, IMG_SIZE, 4), tf.float32),
             'aux':   tf.TensorSpec((AUX_DIM,), tf.float32)},
            tf.TensorSpec((), tf.float32)
        )
    )
    if not augment: ds = ds.cache()
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds_w = make_weight_ds(train_df, augment=True)
val_ds_w   = make_weight_ds(val_df,   augment=False)
test_ds_w  = make_weight_ds(test_df,  augment=False)
print('Weight datasets ready.')

In [ ]:
# ── Stage 3: train all variants and compare ───────────────────────────────────

def train_weight_model(name, backbone_weights_path, train_ds, val_ds, test_ds):
    save_path = OUTPUT_DIR / f'weight_model_{name}.keras'
    if save_path.exists():
        print(f'[{name}] Already trained — loading')
        m = keras.models.load_model(str(save_path),
                                    custom_objects={'mape_metric': mape_metric, 'mae_kg_metric': mae_kg_metric})
        res = m.evaluate(test_ds, verbose=0)
        del m; gc.collect()
        return {'name': name, 'mape': res[1], 'mae': res[2]}

    print(f'\n=== Training variant [{name}] ===')
    model = build_weight_model(backbone_weights_path, freeze_fraction=0.7)

    model.compile(
        optimizer=keras.optimizers.AdamW(1e-3, weight_decay=1e-4, clipnorm=1.0),
        loss=keras.losses.Huber(delta=1.0),
        metrics=[mape_metric, mae_kg_metric]
    )
    model.fit(train_ds, validation_data=val_ds, epochs=15, verbose=1,
              callbacks=[keras.callbacks.EarlyStopping('val_mape_metric', patience=5, restore_best_weights=True)])

    for lyr in model.get_layer('backbone').layers[int(len(model.get_layer('backbone').layers) * 0.6):]:
        lyr.trainable = True
    total_b = 25 * math.ceil(len(train_df) / BATCH_SIZE)
    warmup_b = max(1, int(0.05 * total_b))

    class WarmupCos(keras.optimizers.schedules.LearningRateSchedule):
        def __call__(self, step):
            step = tf.cast(step, tf.float32)
            warmup_lr = 5e-5 * (step / warmup_b)
            cos_lr = 5e-5 * 0.5 * (1 + tf.cos(math.pi * (step - warmup_b) / (total_b - warmup_b)))
            return tf.where(step < warmup_b, warmup_lr, cos_lr)
        def get_config(self): return {}

    model.compile(
        optimizer=keras.optimizers.AdamW(WarmupCos(), weight_decay=1e-4, clipnorm=1.0),
        loss=keras.losses.Huber(delta=1.0),
        metrics=[mape_metric, mae_kg_metric]
    )
    model.fit(train_ds, validation_data=val_ds, epochs=25, verbose=1,
              callbacks=[
                  keras.callbacks.ModelCheckpoint(str(save_path), save_best_only=True, monitor='val_mape_metric'),
                  keras.callbacks.EarlyStopping('val_mape_metric', patience=8, restore_best_weights=True),
              ])

    model = keras.models.load_model(str(save_path),
                                    custom_objects={'mape_metric': mape_metric, 'mae_kg_metric': mae_kg_metric})
    res = model.evaluate(test_ds, verbose=0)
    print(f'[{name}] Test → MAPE={res[1]:.2f}%  MAE={res[2]:.1f} kg')
    del model; gc.collect()
    return {'name': name, 'mape': res[1], 'mae': res[2]}

results = []
results.append(train_weight_model('A_imagenet',   None,                  train_ds_w, val_ds_w, test_ds_w))
results.append(train_weight_model('B_opencows',   BACKBONE_SAVE,         train_ds_w, val_ds_w, test_ds_w))
results.append(train_weight_model('C_ap10k',      CATTLE_KP_SAVE,        train_ds_w, val_ds_w, test_ds_w))
# D: Indian-adapted backbone — same AP-10K keypoints but first seen Indian breeds
if INDIAN_BACKBONE_SAVE.exists():
    results.append(train_weight_model('D_indian',  INDIAN_BACKBONE_SAVE,  train_ds_w, val_ds_w, test_ds_w))
else:
    print('[D_indian] Skipped — run Stage 1.5 first (download_datasets.py --indian)')

In [ ]:
# ── Comparison table ───────────────────────────────────────────────────────────
cmp = pd.DataFrame(results).sort_values('mape')
print('\n' + '='*55)
print('PRETRAINING BENEFIT — TEST SET COMPARISON')
print('='*55)
print(cmp[['name','mape','mae']].to_string(index=False, float_format=lambda x: f'{x:.2f}'))
print('='*55)

baseline_mape = next(r['mape'] for r in results if 'imagenet' in r['name'])
for r in results:
    if 'imagenet' not in r['name']:
        delta = baseline_mape - r['mape']
        print(f"  {r['name']}: {'+' if delta>0 else ''}{delta:.2f}% vs ImageNet baseline")

best = cmp.iloc[0]
print(f"\nBest variant: {best['name']}  MAPE={best['mape']:.2f}%  MAE={best['mae']:.1f} kg")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
cmp.plot.bar(x='name', y='mape', ax=ax1, legend=False, color=['#e74c3c','#2ecc71','#3498db'])
ax1.set_title('MAPE by Init (lower = better)'); ax1.set_ylabel('MAPE (%)')
cmp.plot.bar(x='name', y='mae',  ax=ax2, legend=False, color=['#e74c3c','#2ecc71','#3498db'])
ax2.set_title('MAE by Init (lower = better)'); ax2.set_ylabel('MAE (kg)')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'pretraining_comparison.png'), dpi=120)
plt.show()

In [ ]:
# ── TFLite export of best model ────────────────────────────────────────────────
best_path = OUTPUT_DIR / f"weight_model_{best['name']}.keras"
best_model = keras.models.load_model(str(best_path),
                                     custom_objects={'mape_metric': mape_metric, 'mae_kg_metric': mae_kg_metric})

# FP16
conv = tf.lite.TFLiteConverter.from_keras_model(best_model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
tflite_fp16 = conv.convert()
fp16_path = OUTPUT_DIR / 'cattle_weight_local_fp16.tflite'
fp16_path.write_bytes(tflite_fp16)
print(f'FP16 TFLite: {len(tflite_fp16)/1e6:.2f} MB → {fp16_path}')

# INT8
def rep_dataset():
    for r in val_df.head(100).to_dict('records'):
        try:
            img, aux, _ = load_weight_sample(r, augment=False)
            yield [img[np.newaxis].astype(np.float32), aux[np.newaxis].astype(np.float32)]
        except Exception: pass

conv8 = tf.lite.TFLiteConverter.from_keras_model(best_model)
conv8.optimizations = [tf.lite.Optimize.DEFAULT]
conv8.representative_dataset = rep_dataset
conv8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv8.inference_input_type  = tf.float32
conv8.inference_output_type = tf.float32
try:
    tflite_int8 = conv8.convert()
    int8_path = OUTPUT_DIR / 'cattle_weight_local_int8.tflite'
    int8_path.write_bytes(tflite_int8)
    print(f'INT8 TFLite: {len(tflite_int8)/1e6:.2f} MB → {int8_path}')
except Exception as e:
    print(f'INT8 failed ({e}) — FP16 is the deployment model.')

# Save normalization constants
norm_info = {
    'weight_mean': WEIGHT_MEAN, 'weight_std': WEIGHT_STD,
    'img_size': IMG_SIZE, 'aux_dim': AUX_DIM, 'meas_dim': MEAS_DIM,
    'img_mean': IMG_MEAN, 'img_std': IMG_STD,
    'best_variant': best['name'],
    'mape_test': float(best['mape']), 'mae_test_kg': float(best['mae']),
    'pretraining_results': [{'name':r['name'],'mape':float(r['mape']),'mae':float(r['mae'])} for r in results]
}
with open(OUTPUT_DIR / 'normalization_info.json', 'w') as f:
    json.dump(norm_info, f, indent=2)

print(f'\nAll outputs in {OUTPUT_DIR}')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')